# Pipeline conversacional de Chatbot com NLP (chatbot_nltk)

- **Repositório:** `nlp-llms-portfolio`  
- **Tecnologias:** NLTK, spaCy, Scikit-Learn  
- **Arquivo Original:** `Chatbot_nltk.ipynb`  

---


In [6]:
!pip install gensim

In [7]:
!pip install numpy==1.23.5

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 92.8 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
imbalanced-learn 0.13.0 requires numpy<3,>=1.24.3, but you have numpy 1.23.5 which is incompatible.
jaxlib 0.5.1 requires numpy>=1.25, but you have numpy 1.23.5 which is incompatible.
pymc 5.22.0 requires numpy>=1.25.0, but you have numpy 1.23.5 which is incompatible.
xarray 2025.3.1 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
blosc2 3.3.3 requires numpy>=1.26, but you have numpy 1.23.5 which is incompatible.
scikit-image 0.25.2 requires numpy>=1.24, but you have numpy 1.23.5 which is incompatible.
chex 0.1.89 requires numpy>=1.24.1, but you have numpy 1.23.5 wh

In [8]:
import numpy as np
import re
import gensim
from gensim.models import Word2Vec
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import RSLPStemmer

ModuleNotFoundError: No module named 'numpy.rec'

In [ ]:
# Baixar recursos do NLTK (apenas uma vez)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('rslp')  # Para o stemmer em português

In [ ]:
with open("o-guarani.txt", "r", encoding="utf-8") as f:
    raw = f.read()

In [ ]:
# Limpeza básica
raw = re.sub(r"\s+", " ", raw)  # Remove espaços extras
text = raw.lower()

In [ ]:
# Tokenização e pré-processamento com NLTK
stop_words = set(stopwords.words('portuguese'))
stemmer = RSLPStemmer()


# Tokenização por sentenças para o Word2Vec
sentences = []
for sent in sent_tokenize(text, language='portuguese'):
    tokens = [stemmer.stem(tok) for tok in word_tokenize(sent, language='portuguese')
              if tok.isalpha() and tok not in stop_words]
    if tokens:  # Só adiciona se não estiver vazio
        sentences.append(tokens)

In [ ]:
# Tokenização simples para os chunks
tokens = [tok for tok in word_tokenize(text, language='portuguese') if tok.isalpha()]

print(f"Total de tokens limpos: {len(tokens)}")
print(tokens[:20])

In [ ]:
# Configura e treina o Word2Vec
w2v = Word2Vec(
    sentences,
    vector_size=100,    # dimensão do embedding
    window=5,           # contexto
    min_count=2,
    sg=1,               # Skip-gram
    epochs=10,
    workers=2
)

# Salva modelo
w2v.save("w2v_o_guarani_nltk.model")
print("Word2Vec treinado e salvo.")

In [ ]:
# Criação dos chunks
chunk_size = 250
step = chunk_size // 2

chunk_texts = []  # textos originais para exibir na resposta
chunks = []       # lista de tokens para vetores

for start in range(0, len(tokens), step):
    end = start + chunk_size
    chunk = tokens[start:end]
    if len(chunk) < 50:
        break
    chunks.append(chunk)
    chunk_text = " ".join(chunk)
    chunk_texts.append(chunk_text)

print(f"Gerados {len(chunks)} chunks de ~{chunk_size} tokens.")

# Pré-cálcula vetores médios dos chunks
chunk_vecs = []
for chunk in chunks:
    vecs = [w2v.wv[stemmer.stem(w)] for w in chunk if stemmer.stem(w) in w2v.wv]
    if vecs:
        chunk_vecs.append(np.mean(vecs, axis=0))
    else:
        chunk_vecs.append(np.zeros(w2v.vector_size))


In [ ]:
# Função de resposta modificada para NLTK
from numpy.linalg import norm
from scipy.spatial.distance import cosine

def responde(pergunta, top_k=1):
    # Pré-processa a pergunta com NLTK
    q_tokens = [stemmer.stem(tok) for tok in word_tokenize(pergunta.lower(), language='portuguese')
               if tok.isalpha() and tok not in stop_words and stemmer.stem(tok) in w2v.wv]

    if not q_tokens:
        return "Não entendi — use palavras do livro."

    q_vec = np.mean([w2v.wv[w] for w in q_tokens], axis=0)

    # Calcula similaridades
    sims = [1 - cosine(q_vec, cv) for cv in chunk_vecs]
    idxs = np.argsort(sims)[::-1][:top_k]

    respostas = []
    for i in idxs:
        texto = chunk_texts[i]
        respostas.append((sims[i], texto[:500] + "..."))  # mostra 500 chars
    return respostas

# Interface (mantida igual)
import ipywidgets as widgets
from IPython.display import display, clear_output

input_pergunta = widgets.Text(
    placeholder="Digite sua pergunta sobre O Guarani",
    description="Pergunta:",
    layout=widgets.Layout(width="80%")
)
botao = widgets.Button(description="Responder")
saida = widgets.Output()

def on_click(b):
    with saida:
        clear_output()
        res = responde(input_pergunta.value, top_k=1)
        if isinstance(res, str):
            print(res)
        else:
            score, trecho = res[0]
            print(f"Similaridade: {score:.3f}\n\n{trecho}")

botao.on_click(on_click)

display(input_pergunta, botao, saida)